<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/AHA_sis3_time_dependence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Slide 3 / Figure 3 — SIS3 status varies by time since stroke
# Colab-ready script
#
# Input file expected in the current Colab folder (/content):
#   C_patient_data*.csv
#
# Output folder:
#   AHA_slide03_outputs/
# ============================================================

from pathlib import Path
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import PercentFormatter

# -----------------------------
# User settings
# -----------------------------
BASE = Path.cwd()  # Colab default: /content
OUT = BASE / "AHA_slide03_outputs_v6"
OUT.mkdir(parents=True, exist_ok=True)

SIS3_CUTOFF = 63
RANDOM_SEED = 42

FIGURE_TITLE = "SIS3 Status and Distributions Vary by Time Since Stroke"

COLORS = {
    "lower": "#D55E00",     # consistent across slides: lower SIS3 = orange
    "higher": "#0072B2",    # consistent across slides: higher SIS3 = blue
    "black": "#303030",
    "gray": "#6E6E6E",
    "lightgray": "#D9D9D9",
    "verylight": "#EAEAEA",
    "violin": "#D8E3EA",
}

TIME_BINS = [
    ("0–1y", 0.0, 1.0),
    ("1–3y", 1.0, 3.0),
    (">3y", 3.0, np.inf),
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 8.8,
    "legend.fontsize": 8.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 450,
})

# -----------------------------
# Helper functions
# -----------------------------
def read_csv_safely(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            df.columns = df.columns.astype(str).str.strip()
            return df
        except UnicodeDecodeError:
            pass
    df = pd.read_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    return df


def csv_files():
    return sorted(BASE.glob("*.csv"))


def find_csv(contains_all=None, contains_any=None, required=True, label="file"):
    contains_all = [x.lower() for x in (contains_all or [])]
    contains_any = [x.lower() for x in (contains_any or [])]
    matches = []
    for f in csv_files():
        name = f.name.lower()
        if contains_all and not all(x in name for x in contains_all):
            continue
        if contains_any and not any(x in name for x in contains_any):
            continue
        matches.append(f)
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not find {label}. Tried all={contains_all}, any={contains_any}")
        return None
    return sorted(matches, key=lambda p: (len(p.name), p.name))[0]


def find_col(df, candidates, required=True, label="column"):
    lower_to_col = {c.lower(): c for c in df.columns}
    for cand in candidates:
        c = lower_to_col.get(cand.lower())
        if c is not None:
            return c
    for c in df.columns:
        c_l = c.lower()
        if any(cand.lower() in c_l for cand in candidates):
            return c
    if required:
        raise ValueError(f"Could not find {label}. Tried {candidates}. Available columns: {df.columns.tolist()}")
    return None


def detect_time_years(series, colname=""):
    x = pd.to_numeric(series, errors="coerce")
    lname = str(colname).lower()
    max_val = np.nanmax(x.values) if np.isfinite(x).any() else np.nan
    if "year" in lname or "yrs" in lname or "yr" in lname:
        return x
    if "day" in lname or (np.isfinite(max_val) and max_val > 40):
        return x / 365.25
    if "month" in lname or (np.isfinite(max_val) and max_val > 6):
        return x / 12.0
    return x


def assign_time_bin(time_years):
    for label, lo, hi in TIME_BINS:
        if time_years >= lo and time_years < hi:
            return label
    return np.nan


def wilson_ci(k, n, z=1.96):
    """Wilson 95% CI for a binomial proportion."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p * (1 - p) + z**2 / (4*n)) / n) / denom
    return center - half, center + half


def fmt_pct(x):
    if pd.isna(x):
        return "NA"
    return f"{100*x:.1f}"


def wrap_caption(text, width=170):
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


# -----------------------------
# 1. Load and prepare clinical data
# -----------------------------
print("Current folder:", BASE)
print("CSV files found:")
for f in csv_files():
    print(" -", f.name)

patient_file = find_csv(contains_any=["patient"], required=True, label="patient file")
patient = read_csv_safely(patient_file)

pid_col = find_col(patient, ["pid", "patient_id", "participant_id", "subject_id", "id"], required=False, label="patient ID column")
sis3_col = find_col(patient, ["sis3", "sis_3", "sis 3"], label="SIS3 column")
time_col = find_col(patient, ["timesince", "time_since", "time", "days", "months", "years"], label="time-since-stroke column")

df = patient.copy()
df["sis3"] = pd.to_numeric(df[sis3_col], errors="coerce")
df["time_years"] = detect_time_years(df[time_col], time_col)
df = df.dropna(subset=["sis3", "time_years"]).copy()
df["time_bin"] = df["time_years"].map(assign_time_bin)
df = df.dropna(subset=["time_bin"]).copy()
df["sis3_group"] = np.where(df["sis3"] <= SIS3_CUTOFF, "Lower SIS3 (≤63)", "Higher SIS3 (>63)")
df["is_lower_sis3"] = df["sis3"] <= SIS3_CUTOFF

time_order = [x[0] for x in TIME_BINS]
df["time_bin"] = pd.Categorical(df["time_bin"], categories=time_order, ordered=True)
df = df.sort_values(["time_bin", "sis3"]).reset_index(drop=True)

# -----------------------------
# 2. Summary table for backup
# -----------------------------
rows = []
for label in time_order:
    sub = df[df["time_bin"] == label]
    n = len(sub)
    k = int(sub["is_lower_sis3"].sum())
    prop = k / n if n else np.nan
    ci_low, ci_high = wilson_ci(k, n)
    rows.append({
        "Time since stroke": label,
        "No. of patients": n,
        "Lower SIS3, n": k,
        "Higher SIS3, n": n - k,
        "Lower SIS3, %": 100 * prop,
        "Lower SIS3, 95% CI lower": 100 * ci_low,
        "Lower SIS3, 95% CI upper": 100 * ci_high,
        "Lower SIS3, % (95% CI)": f"{100*prop:.1f} ({100*ci_low:.1f} to {100*ci_high:.1f})" if n else "NA",
    })

summary = pd.DataFrame(rows)
df.to_csv(OUT / "slide03_patient_time_sis3_used.csv", index=False)
summary.to_csv(OUT / "slide03_lower_sis3_prevalence_by_time.csv", index=False)

# -----------------------------
# 3. Journal-style Figure 3
# -----------------------------
fig = plt.figure(figsize=(14.0, 8.2))
gs = GridSpec(
    3, 3,
    figure=fig,
    height_ratios=[0.42, 4.15, 1.75],
    width_ratios=[0.95, 0.90, 0.90],
    hspace=0.18,
    wspace=0.26,
)

# Title
title_ax = fig.add_subplot(gs[0, :])
title_ax.axis("off")
title_ax.text(
    0.00, 0.72,
    "Figure 3. " + FIGURE_TITLE,
    ha="left", va="center", fontsize=13.0, fontweight="bold",
    color=COLORS["black"], transform=title_ax.transAxes,
)

# Panel A: stacked prevalence bar plot
ax1 = fig.add_subplot(gs[1, 0])
x = np.arange(len(time_order))
bar_width = 0.52

lower_counts = summary["Lower SIS3, n"].values.astype(float)
higher_counts = summary["Higher SIS3, n"].values.astype(float)
totals = summary["No. of patients"].values.astype(float)

lower_prop = lower_counts / totals
higher_prop = higher_counts / totals

ax1.bar(x, lower_prop, width=bar_width, color=COLORS["lower"], label="Lower SIS3 (≤63)")
ax1.bar(x, higher_prop, width=bar_width, bottom=lower_prop, color=COLORS["higher"], label="Higher SIS3 (>63)")

for i, (lp, hp, lc, hc, n) in enumerate(zip(lower_prop, higher_prop, lower_counts, higher_counts, totals)):
    ax1.text(i, 1.035, f"n={int(n)}", ha="center", va="bottom", fontsize=8.8, color=COLORS["black"])
    if lp > 0.04:
        ax1.text(i, lp / 2, f"{lp*100:.1f}%\n(n={int(lc)})", ha="center", va="center", fontsize=8.5, color="white")
    else:
        ax1.text(i, lp + 0.03, f"{lp*100:.1f}% (n={int(lc)})", ha="center", va="bottom", fontsize=8.2, color=COLORS["black"])
    if hp > 0.08:
        ax1.text(i, lp + hp / 2, f"{hp*100:.1f}%\n(n={int(hc)})", ha="center", va="center", fontsize=8.5, color="white")

ax1.set_xticks(x)
ax1.set_xticklabels(time_order)
ax1.set_ylim(0, 1.06)
ax1.set_ylabel("Proportion of patients")
ax1.set_xlabel("Time since stroke")
ax1.yaxis.set_major_formatter(PercentFormatter(1.0))
ax1.set_title("A. Prevalence of lower SIS3 by time since stroke", loc="left", pad=8, fontweight="bold")
ax1.set_axisbelow(True)
ax1.grid(axis="y", color=COLORS["verylight"], lw=0.7)
ax1.tick_params(axis="both", length=3, color=COLORS["gray"])
ax1.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.00, -0.10), ncol=1, handlelength=1.4, borderaxespad=0.0)

# Panel B: split distributions
ax2_low = fig.add_subplot(gs[1, 1])
ax2_high = fig.add_subplot(gs[1, 2], sharey=ax2_low)

def draw_group_distribution(ax, group_label, color, title_text, show_ylabel=False):
    sub_group = df[df["sis3_group"] == group_label].copy()
    data_by_bin = [sub_group.loc[sub_group["time_bin"] == label, "sis3"].values for label in time_order]
    positions = np.arange(len(time_order))

    valid = [arr for arr in data_by_bin if len(arr) > 0]
    pos_valid = [i for i, arr in enumerate(data_by_bin) if len(arr) > 0]
    if valid:
        parts = ax.violinplot(
            valid,
            positions=pos_valid,
            widths=0.72,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body in parts["bodies"]:
            body.set_facecolor(COLORS["violin"])
            body.set_edgecolor("none")
            body.set_alpha(0.85)

    rng = np.random.default_rng(RANDOM_SEED + (1 if "Higher" in group_label else 0))
    for i, label in enumerate(time_order):
        sub = sub_group[sub_group["time_bin"] == label].copy()
        jitter = rng.normal(0, 0.05, size=len(sub))
        if len(sub) > 0:
            ax.scatter(
                np.full(len(sub), i) + jitter,
                sub["sis3"],
                s=24,
                color=color,
                alpha=0.92,
                edgecolor="white",
                linewidth=0.35,
                zorder=3,
            )
            median = sub["sis3"].median()
            q1 = sub["sis3"].quantile(0.25)
            q3 = sub["sis3"].quantile(0.75)
            ax.plot([i - 0.20, i + 0.20], [median, median], color=COLORS["black"], lw=1.2, zorder=4)
            ax.plot([i, i], [q1, q3], color=COLORS["black"], lw=1.0, zorder=4)
        ax.text(i, 102.0, f"n={len(sub)}", ha="center", va="bottom", fontsize=8.7, color=COLORS["black"])

    ax.axhline(SIS3_CUTOFF, color=COLORS["gray"], lw=0.9, ls="--")
    ax.set_xlim(-0.55, len(time_order) - 0.45)
    ax.set_ylim(0, 106)
    ax.set_xticks(positions)
    ax.set_xticklabels(time_order)
    ax.set_xlabel("Time since stroke")
    if show_ylabel:
        ax.set_ylabel("SIS3")
    ax.set_title(title_text, pad=4, fontweight="bold")
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=COLORS["verylight"], lw=0.7)
    ax.tick_params(axis="both", length=3, color=COLORS["gray"])

draw_group_distribution(ax2_low, "Lower SIS3 (≤63)", COLORS["lower"], "B1. Lower SIS3 only", show_ylabel=True)
draw_group_distribution(ax2_high, "Higher SIS3 (>63)", COLORS["higher"], "B2. Higher SIS3 only", show_ylabel=True)
ax2_high.tick_params(axis="y", labelleft=True, left=True)

ax2_high.text(2.42, SIS3_CUTOFF - 2.0, "SIS3 = 63", ha="right", va="top", fontsize=8.5, color=COLORS["gray"])

# Caption
cap_ax = fig.add_subplot(gs[2, :])
cap_ax.axis("off")
caption = (
    "Lower SIS3 was defined as SIS3 ≤63 and higher SIS3 as SIS3 >63. Time since stroke was grouped as 0–1 year, 1–3 years, and more than 3 years. "
    "Panel A shows the proportion of patients in each SIS3 group within each time bin, with total sample size shown above each bar. Panels B1 and B2 show "
    "SIS3 distributions separately for lower-SIS3 and higher-SIS3 patients, with sample sizes labeled within each time bin; horizontal black lines "
    "indicate medians and vertical black lines indicate interquartile ranges. Orange indicates lower SIS3 and blue indicates higher SIS3, consistent with Figures 1 and 2."
)
cap_ax.text(
    0.00, 0.68,
    wrap_caption(caption, width=182),
    ha="left", va="top", fontsize=8.6, color=COLORS["black"],
    linespacing=1.22, transform=cap_ax.transAxes,
)

fig.subplots_adjust(left=0.06, right=0.98, top=0.93, bottom=0.08)

for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT / f"figure3_sis3_time_dependence_journal_style_v6.{ext}", bbox_inches="tight")
plt.close(fig)

print("\nCreated Slide 3 outputs:")
for p in sorted(OUT.glob("figure3_sis3_time_dependence_journal_style_v6.*")):
    print(" -", p)
print("\nBackup summary table:")
print(summary.to_string(index=False))



Current folder: /content
CSV files found:
 - C_NPX_data.csv
 - C_patient_data.csv
 - NAME_OID.csv
 - strokecog_literature_aligned_pathway_framework.csv
 - strokecog_literature_aligned_protein_pathway_assignment_summary.csv

Created Slide 3 outputs:
 - /content/AHA_slide03_outputs_v4/figure3_sis3_time_dependence_journal_style_v4.pdf
 - /content/AHA_slide03_outputs_v4/figure3_sis3_time_dependence_journal_style_v4.png
 - /content/AHA_slide03_outputs_v4/figure3_sis3_time_dependence_journal_style_v4.svg

Backup summary table:
Time since stroke  No. of patients  Lower SIS3, n  Higher SIS3, n  Lower SIS3, %  Lower SIS3, 95% CI lower  Lower SIS3, 95% CI upper Lower SIS3, % (95% CI)
             0–1y               47              6              41      12.765957                  5.984523                 25.174213     12.8 (6.0 to 25.2)
             1–3y               15              4              11      26.666667                 10.897276                 51.950890    26.7 (10.9 to 52.0)
     